# Trade EDA and concentration metrics

This notebook uses the repository's cleaned bilateral trade matrix. It computes import-partner shares, CR1/CR2, HHI, Shannon diversity, correlations, and partner trends.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

here = Path.cwd().resolve()
repo_root = next((p for p in (here, *here.parents) if (p / "data" / "cleaned").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Could not locate data/cleaned from the notebook working directory")

matrix_path = repo_root / "data" / "cleaned" / "trade_matrix_cleaned.csv"
if not matrix_path.exists():
    raise FileNotFoundError(f"Missing cleaned trade matrix: {matrix_path}")

cleaned_data = pd.read_csv(matrix_path, usecols=[
    "Reporter Countries", "Partner Countries", "Element", "Year", "Unit", "Value"
])
cleaned_data = cleaned_data[
    (cleaned_data["Element"] == "Import value")
    & (cleaned_data["Unit"] == "1000 USD")
    & (cleaned_data["Value"] > 0)
].rename(columns={
    "Reporter Countries": "reporter_country",
    "Partner Countries": "partner_country",
    "Year": "year",
    "Value": "trade_value",
}).copy()
print(f"Loaded {len(cleaned_data):,} positive import-value rows")
cleaned_data.head()

## Import partner concentration

In [ ]:
partner_share = cleaned_data.groupby("partner_country")["trade_value"].sum()
partner_share = partner_share / partner_share.sum()
ranked_shares = partner_share.sort_values(ascending=False)
CR1 = ranked_shares.iloc[0]
CR2 = ranked_shares.iloc[:2].sum()
HHI = (partner_share ** 2).sum()
shannon_index = -np.sum(partner_share[partner_share > 0] * np.log(partner_share[partner_share > 0]))
print(f"CR1: {CR1:.4f}")
print(f"CR2: {CR2:.4f}")
print(f"HHI: {HHI:.4f}")
print(f"Shannon Diversity Index: {shannon_index:.4f}")
ranked_shares.head(10).rename("import_share")

In [ ]:
cleaned_data["HHI"] = HHI
cleaned_data["Shannon_Index"] = shannon_index
corr_matrix = cleaned_data[["year", "trade_value", "HHI", "Shannon_Index"]].corr()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
partner_year = (cleaned_data.groupby(["partner_country", "year"], as_index=False)["trade_value"].sum())
partner_year["YoY_Growth"] = partner_year.groupby("partner_country")["trade_value"].pct_change()
top_partners = partner_year.groupby("partner_country")["trade_value"].sum().nlargest(10).index
plot_data = partner_year[partner_year["partner_country"].isin(top_partners)]
sns.lineplot(data=plot_data, x="year", y="YoY_Growth", hue="partner_country")
plt.title("Year-over-Year Growth Rates for Top Import Partners")
plt.tight_layout()
plt.show()